#### Importing Libraries

In [1]:
# Importing libraries for data preparation
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

#### Loading the Dataset

In [2]:
# Loading the original product classification dataset
df = pd.read_csv("../data/raw/pricerunner_aggregate.csv")

# Displaying the first 10 records
df.head(10)

,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
5,6,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,6,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
6,7,apple iphone 8 plus 64 gb space grey,7,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
7,8,apple iphone 8 plus 64gb space grey,8,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
8,9,apple iphone 8 plus 64gb space grey,9,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
9,10,apple iphone 8 plus 64gb space grey,10,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones


#### Inspecting the Dataset Structure

In [3]:
# Displaying the dataset dimensions
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

# Displaying column names and data types
print("\nDataset Information:")
df.info()

Number of rows: 35,311
Number of columns: 7

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Product ID       35311 non-null  int64
 1   Product Title    35311 non-null  str  
 2    Merchant ID     35311 non-null  int64
 3    Cluster ID      35311 non-null  int64
 4    Cluster Label   35311 non-null  str  
 5    Category ID     35311 non-null  int64
 6    Category Label  35311 non-null  str  
dtypes: int64(4), str(3)
memory usage: 4.9 MB


**Observations on Dataset Structure:**

- The dataset contains **35,311 product listings** and **7 columns**, consistent with the original data.
- All columns contain **35,311 non-null values**, indicating no missing data.
- The dataset includes **4 integer columns** and **3 text columns**, with leading spaces present in several column names.

#### Standardizing Column Names

In [4]:
# Standardizing column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Displaying the standardized column names
df.columns.tolist()

['Product_ID',
 'Product_Title',
 'Merchant_ID',
 'Cluster_ID',
 'Cluster_Label',
 'Category_ID',
 'Category_Label']

#### Identifying Duplicate Records

In [5]:
# Identifying exact duplicate records
exact_duplicates = df.duplicated().sum()

# Identifying duplicate records excluding the product identifier
duplicate_listings = df.duplicated(
    subset=["Product_Title", "Merchant_ID", "Cluster_ID", "Category_ID"]
).sum()

# Identifying repeated product titles
repeated_titles = df.duplicated(subset=["Product_Title"]).sum()

# Displaying duplicate counts
print(f"Exact duplicate records: {exact_duplicates:,}")
print(f"Duplicate listings excluding Product_ID: {duplicate_listings:,}")
print(f"Repeated product title occurrences: {repeated_titles:,}")

Exact duplicate records: 0
Duplicate listings excluding Product_ID: 0
Repeated product title occurrences: 4,318


**Observations on Duplicate Records:**

- No exact duplicate records or duplicate listings excluding `Product_ID` were identified.
- **4,318 repeated product title occurrences** were detected, reflecting identical titles across different product records.
- Repeated titles will be retained at this stage, with their cluster relationships considered during leakage-aware data splitting.

#### Inspecting Product Title Whitespace

In [6]:
# Identifying leading and trailing whitespace
leading_whitespace = df["Product_Title"].str.match(r"^\s").sum()
trailing_whitespace = df["Product_Title"].str.contains(r"\s$").sum()

# Identifying consecutive whitespace characters
repeated_whitespace = df["Product_Title"].str.contains(r"\s{2,}").sum()

# Identifying empty or whitespace-only titles
empty_titles = df["Product_Title"].str.strip().eq("").sum()

# Displaying whitespace-related findings
print(f"Titles with leading whitespace: {leading_whitespace:,}")
print(f"Titles with trailing whitespace: {trailing_whitespace:,}")
print(f"Titles with repeated internal whitespace: {repeated_whitespace:,}")
print(f"Empty or whitespace-only titles: {empty_titles:,}")

Titles with leading whitespace: 0
Titles with trailing whitespace: 0
Titles with repeated internal whitespace: 0
Empty or whitespace-only titles: 0


**Observations on Product Title Whitespace:**

- No product titles contain leading, trailing, or repeated internal whitespace.
- No empty or whitespace-only product titles were identified.
- No whitespace corrections are required, allowing us to preserve the original product descriptions.

#### Inspecting Product Title Capitalization

In [7]:
# Identifying titles containing uppercase letters
uppercase_titles = df["Product_Title"].str.contains(r"[A-Z]").sum()

# Identifying titles containing lowercase letters
lowercase_titles = df["Product_Title"].str.contains(r"[a-z]").sum()

# Calculating capitalization percentages
uppercase_percentage = uppercase_titles / len(df) * 100

# Displaying capitalization findings
print(f"Titles containing uppercase letters: {uppercase_titles:,}")
print(f"Titles containing lowercase letters: {lowercase_titles:,}")
print(f"Percentage containing uppercase letters: {uppercase_percentage:.2f}%")

Titles containing uppercase letters: 0
Titles containing lowercase letters: 35,311
Percentage containing uppercase letters: 0.00%


**Observations on Product Title Capitalization:**

- All **35,311 product titles** contain lowercase letters.
- No uppercase characters were identified across the dataset.
- Lowercase normalization is unnecessary, allowing the existing text to be retained without modification.

#### Identifying Conflicting Category Labels

In [8]:
# Identifying product titles associated with multiple category labels
title_category_counts = (
    df.groupby("Product_Title")["Category_Label"]
    .nunique()
)

# Selecting titles with conflicting category assignments
conflicting_titles = title_category_counts[
    title_category_counts > 1
].index

# Retrieving the affected records
conflicting_records = (
    df[df["Product_Title"].isin(conflicting_titles)]
    [["Product_ID", "Product_Title", "Cluster_ID", "Category_ID", "Category_Label"]]
    .sort_values("Product_Title")
    .reset_index(drop=True)
)

# Displaying the conflicting records
print(f"Conflicting product titles: {len(conflicting_titles):,}")
print(f"Affected records: {len(conflicting_records):,}")

display(conflicting_records)

Conflicting product titles: 2
Affected records: 4


,Product_ID,Product_Title,Cluster_ID,Category_ID,Category_Label
0,43365,blomberg kgm9550 fridge freezer,46085,2622,Fridge Freezers
1,47235,blomberg kgm9550 fridge freezer,47395,2623,Fridges
2,34378,bosch kur15a50gb integrated undercounter fridge,42819,2620,Washing Machines
3,43394,bosch kur15a50gb integrated undercounter fridge,46113,2623,Fridges


**Observations on Conflicting Category Labels:**

- **2 product titles**, affecting **4 records**, have inconsistent category assignments.
- `blomberg kgm9550 fridge freezer` is classified under both `Fridge Freezers` and `Fridges`.
- `bosch kur15a50gb integrated undercounter fridge` is classified under both `Washing Machines` and `Fridges`, indicating a likely labeling error.

#### Inspecting Clusters with Conflicting Category Assignments

In [9]:
# Identifying clusters associated with conflicting product titles
affected_cluster_ids = conflicting_records["Cluster_ID"].unique()

# Retrieving all listings belonging to the affected clusters
affected_clusters = (
    df[df["Cluster_ID"].isin(affected_cluster_ids)]
    [["Product_ID", "Product_Title", "Cluster_ID", "Cluster_Label", "Category_Label"]]
    .sort_values(["Cluster_ID", "Product_Title"])
    .reset_index(drop=True)
)

# Summarizing the affected clusters
cluster_summary = (
    affected_clusters.groupby(["Cluster_ID", "Category_Label"])
    .agg(
        Listing_Count=("Product_ID", "size"),
        Unique_Titles=("Product_Title", "nunique")
    )
    .reset_index()
)

print(f"Affected clusters: {len(affected_cluster_ids)}")
display(cluster_summary)

Affected clusters: 4


,Cluster_ID,Category_Label,Listing_Count,Unique_Titles
0,42819,Washing Machines,1,1
1,46085,Fridge Freezers,1,1
2,46113,Fridges,10,9
3,47395,Fridges,1,1


**Observations on Clusters with Conflicting Category Assignments:**

- Three affected clusters (`42819`, `46085`, and `47395`) contain only **1 listing each**, while cluster `46113` contains **10 listings**.
- The `Bosch KUR15A50GB` title appears in a singleton `Washing Machines` cluster and a larger `Fridges` cluster, supporting the possibility of an incorrect category assignment.
- The `Blomberg KGM9550` title appears in separate singleton clusters with different categories, requiring further inspection before resolving the inconsistency.

#### Inspecting Product Listings Within Affected Clusters

In [10]:
# Displaying product listings within the affected clusters
for cluster_id in sorted(affected_cluster_ids):
    cluster_records = affected_clusters[
        affected_clusters["Cluster_ID"] == cluster_id
    ]

    print(f"\nCluster ID: {cluster_id}")
    print(f"Category: {cluster_records['Category_Label'].iloc[0]}")
    print(f"Cluster Label: {cluster_records['Cluster_Label'].iloc[0]}")
    print(f"Number of Listings: {len(cluster_records)}")

    display(
        cluster_records[
            ["Product_ID", "Product_Title", "Cluster_Label"]
        ].reset_index(drop=True)
    )


Cluster ID: 42819
Category: Washing Machines
Cluster Label: Hotpoint WMUD962G
Number of Listings: 1


,Product_ID,Product_Title,Cluster_Label
0,34378,bosch kur15a50gb integrated undercounter fridge,Hotpoint WMUD962G



Cluster ID: 46085
Category: Fridge Freezers
Cluster Label: Blomberg KGM 9550 White
Number of Listings: 1


,Product_ID,Product_Title,Cluster_Label
0,43365,blomberg kgm9550 fridge freezer,Blomberg KGM 9550 White



Cluster ID: 46113
Category: Fridges
Cluster Label: Bosch KUR15A50GB Integrated
Number of Listings: 10


,Product_ID,Product_Title,Cluster_Label
0,43392,bosch kur15a50gb built in under larder fridge ...,Bosch KUR15A50GB Integrated
1,43400,bosch kur15a50gb built under fridge larder 141...,Bosch KUR15A50GB Integrated
2,43399,bosch kur15a50gb classixx integrated 82cm unde...,Bosch KUR15A50GB Integrated
3,43401,bosch kur15a50gb classixx integrated 82cm unde...,Bosch KUR15A50GB Integrated
4,43402,bosch kur15a50gb integrated built under fridge...,Bosch KUR15A50GB Integrated
5,43398,bosch kur15a50gb integrated under counter fridge,Bosch KUR15A50GB Integrated
6,43394,bosch kur15a50gb integrated undercounter fridge,Bosch KUR15A50GB Integrated
7,43395,bosch kur15a50gb/kur15a50gb,Bosch KUR15A50GB Integrated
8,43396,bosch serie 6 built under larder fridge in white,Bosch KUR15A50GB Integrated
9,43397,bosch serie 6 kur15a50gb classixx 60cm wide in...,Bosch KUR15A50GB Integrated



Cluster ID: 47395
Category: Fridges
Cluster Label: Blomberg KGM9550 Stainless Steel
Number of Listings: 1


,Product_ID,Product_Title,Cluster_Label
0,47235,blomberg kgm9550 fridge freezer,Blomberg KGM9550 Stainless Steel


**Observations on Product Listings Within Affected Clusters:**

- Cluster `42819` contains a Bosch refrigerator title incorrectly associated with a `Hotpoint WMUD962G` washing machine cluster, indicating a mismatched product listing.
- Cluster `46113` consistently represents the `Bosch KUR15A50GB` refrigerator, with 10 listings assigned to `Fridges`.
- Both Blomberg clusters represent variants of the `KGM9550` fridge freezer, suggesting that the `Fridges` assignment in cluster `47395` is inconsistent with the product type.

#### Resolving Conflicting Category Assignments

In [11]:
# Creating a copy before applying data quality corrections
df_clean = df.copy()

# Removing the mismatched Bosch product listing
df_clean = df_clean[
    df_clean["Product_ID"] != 34378
].copy()

# Correcting the Blomberg fridge freezer category assignment
blomberg_mask = df_clean["Product_ID"] == 47235

df_clean.loc[blomberg_mask, "Category_ID"] = 2622
df_clean.loc[blomberg_mask, "Category_Label"] = "Fridge Freezers"

# Resetting the dataframe index
df_clean.reset_index(drop=True, inplace=True)

# Verifying the corrected records
print(f"Original records: {len(df):,}")
print(f"Cleaned records: {len(df_clean):,}")
print(f"Records removed: {len(df) - len(df_clean):,}")

display(
    df_clean[
        df_clean["Product_ID"].isin([34378, 43365, 43394, 47235])
    ][["Product_ID", "Product_Title", "Category_ID", "Category_Label"]]
)

Original records: 35,311
Cleaned records: 35,310
Records removed: 1


C:\Users\user\AppData\Local\Temp\ipykernel_1856\3716315191.py:16: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.reset_index(drop=True, inplace=True)


,Product_ID,Product_Title,Category_ID,Category_Label
31699,43365,blomberg kgm9550 fridge freezer,2622,Fridge Freezers
31727,43394,bosch kur15a50gb integrated undercounter fridge,2623,Fridges
35195,47235,blomberg kgm9550 fridge freezer,2622,Fridge Freezers


#### Validating Category Label Consistency

In [12]:
# Checking whether identical product titles have conflicting category labels
remaining_conflicts = (
    df_clean.groupby("Product_Title")["Category_Label"]
    .nunique()
    .gt(1)
    .sum()
)

# Checking consistency between category IDs and category labels
category_id_conflicts = (
    df_clean.groupby("Category_ID")["Category_Label"]
    .nunique()
    .gt(1)
    .sum()
)

category_label_conflicts = (
    df_clean.groupby("Category_Label")["Category_ID"]
    .nunique()
    .gt(1)
    .sum()
)

# Displaying validation results
print(f"Titles with conflicting categories: {remaining_conflicts}")
print(f"Category IDs mapped to multiple labels: {category_id_conflicts}")
print(f"Category labels mapped to multiple IDs: {category_label_conflicts}")
print(f"Cleaned dataset dimensions: {df_clean.shape}")

Titles with conflicting categories: 0
Category IDs mapped to multiple labels: 0
Category labels mapped to multiple IDs: 0
Cleaned dataset dimensions: (35310, 7)
